# 🧬 Preparar Dataset RGI — *K. pneumoniae* + Ertapenem

**Qué hace este notebook (en orden):**
1. Monta Drive
2. Mueve archivos nuevos a `dataset_limpio/` sin duplicar los que ya existen
3. Filtra archivos vacíos (≤ 500 bytes → RGI no encontró nada)
4. Carga secuencias filtrando solo `Cut_Off = Perfect / Strict`, rastreando cada cepa (`genome_id`)
5. Elimina secuencias con label contradictorio (mismo gen en S y R)
6. Hace el split **por cepa** (no por secuencia) para evitar leakage
7. Auditoría completa y guarda `train_por_cepa.csv` / `val_por_cepa.csv`


## Celda 0 · Montar Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('../data')
    IN_COLAB = True
    print('✅ Google Colab detectado y Drive montado.')
except Exception:
    IN_COLAB = False
    print('ℹ️ Entorno local detectado (no Google Colab).')


Mounted at ../data
✅ Drive montado


## Celda 1 · Configuración de rutas
> Ajusta aquí si tus carpetas tienen nombres distintos.

In [ ]:
import os
from pathlib import Path

# ── CONFIGURACIÓN DE RUTAS ──────────────────────────────────────────────
if 'IN_COLAB' in locals() and IN_COLAB:
    SRC_SUSCEPTIBLE = '../data/rgi_Susceptible'
    SRC_RESISTENTE  = '../data/rgi_Resistente'
    BASE_DESTINO    = '../data'
else:
    REPO_ROOT = Path(os.getcwd()).resolve()
    if REPO_ROOT.name == 'notebooks':
        REPO_ROOT = REPO_ROOT.parent
    SRC_SUSCEPTIBLE = str(REPO_ROOT / 'data' / 'raw_rgi' / 'Susceptible')
    SRC_RESISTENTE  = str(REPO_ROOT / 'data' / 'raw_rgi' / 'Resistente')
    BASE_DESTINO    = str(REPO_ROOT / 'data' / 'dataset_limpio')

DST_SUS       = os.path.join(BASE_DESTINO, 'Susceptible')
DST_RES       = os.path.join(BASE_DESTINO, 'Resistente')
RUTA_SALIDA   = BASE_DESTINO   # donde se guardan los CSV finales

# ── FILTROS ────────────────────────────────────────────────────────────────
MIN_BYTES    = 500    # archivos <= este tamaño se ignoran (RGI sin hits)
CUTOFF_OK    = {'Perfect', 'Strict'}  # solo hits de alta confianza
MIN_SEQ_LEN  = 20    # mínimo de bp para considerar una secuencia válida
VAL_FRACTION = 0.20  # fracción de cepas para validación
RANDOM_SEED  = 42

for d in [DST_SUS, DST_RES]:
    os.makedirs(d, exist_ok=True)

print('✅ Configuración lista')
print(f'   Origen S  → {SRC_SUSCEPTIBLE}')
print(f'   Origen R  → {SRC_RESISTENTE}')
print(f'   Destino   → {BASE_DESTINO}')


✅ Configuración lista
   Origen S  → ../data/rgi_Susceptible
   Origen R  → ../data/rgi_Resistente
   Destino   → ../data


## Celda 2 · Mover archivos nuevos → `dataset_limpio/`

- Copia solo los archivos que **no existen ya** en destino (sin duplicar).
- Descarta directamente los archivos ≤ `MIN_BYTES` bytes (RGI vacío).


In [ ]:
import shutil

def mover_nuevos(origen, destino, min_bytes=MIN_BYTES):
    if not os.path.exists(origen):
        print(f'⚠️  Origen no encontrado: {origen}')
        return
    archivos = [f for f in os.listdir(origen) if f.endswith('.txt')]
    copiados = omitidos_dup = omitidos_vacio = 0
    vacios = []
    for f in archivos:
        src_path = os.path.join(origen, f)
        dst_path = os.path.join(destino, f)
        size = os.path.getsize(src_path)
        if size <= min_bytes:
            omitidos_vacio += 1
            vacios.append((f, size))
            continue
        if os.path.exists(dst_path):
            omitidos_dup += 1
            continue
        shutil.copy2(src_path, dst_path)
        copiados += 1
    print(f'\n📁 {os.path.basename(destino)}')
    print(f'   ✅ Copiados nuevos         : {copiados}')
    print(f'   ⏭  Ya existían (omitidos)  : {omitidos_dup}')
    print(f'   🗑  Vacíos descartados       : {omitidos_vacio} (≤{min_bytes} bytes)')
    if vacios:
        print(f'   Ejemplos vacíos: {[v[0] for v in vacios[:3]]}')
    return copiados

print('=' * 55)
print('MOVIENDO DATOS NUEVOS A DATASET_LIMPIO')
print('=' * 55)
mover_nuevos(SRC_SUSCEPTIBLE, DST_SUS)
mover_nuevos(SRC_RESISTENTE,  DST_RES)
print('=' * 55)
print('¡Proceso finalizado!')


MOVIENDO DATOS NUEVOS A DATASET_LIMPIO

📁 Susceptible
   ✅ Copiados nuevos         : 7
   ⏭  Ya existían (omitidos)  : 2549
   🗑  Vacíos descartados       : 100 (≤500 bytes)
   Ejemplos vacíos: ['Kp_ertapenem_S_47-2_61_573_81910_rgi.txt', 'Kp_ertapenem_S_MRSN972125_573_80604_rgi.txt', 'Kp_ertapenem_S_MRSN110345_573_80606_rgi.txt']

📁 Resistente
   ✅ Copiados nuevos         : 2
   ⏭  Ya existían (omitidos)  : 1871
   🗑  Vacíos descartados       : 14 (≤500 bytes)
   Ejemplos vacíos: ['Kp_ertapenem_R_27097_7#8_573_66424_rgi.txt', 'Kp_ertapenem_R_26994_8#178_573_66418_rgi.txt', 'Kp_ertapenem_R_26994_8#146_573_66409_rgi.txt']
¡Proceso finalizado!


## Celda 3 · Diagnóstico: conteo de archivos y vacíos restantes

Revisamos cuántos archivos hay en `dataset_limpio` y si quedó alguno vacío.


In [ ]:
import re

def diagnostico_carpeta(ruta, label, min_bytes=MIN_BYTES):
    archivos = [f for f in os.listdir(ruta) if f.endswith(('.txt', '.tsv'))]
    vacios   = [f for f in archivos if os.path.getsize(os.path.join(ruta, f)) <= min_bytes]
    utiles   = [f for f in archivos if f not in vacios]
    genome_ids = set()
    for f in utiles:
        m = re.search(r'(GC[AF]_\d+\.\d+)', f)
        if m: genome_ids.add(m.group(1))
    print(f'\n🔬 {label}')
    print(f'   Archivos totales     : {len(archivos)}')
    print(f'   ✅ Con datos (útiles) : {len(utiles)}')
    print(f'   🗑  Vacíos (≤{min_bytes}b)  : {len(vacios)}')
    print(f'   Genomas únicos (GCA/F): {len(genome_ids)}')
    return utiles, vacios

print('=' * 55)
utiles_s, vacios_s = diagnostico_carpeta(DST_SUS, 'SUSCEPTIBLES')
utiles_r, vacios_r = diagnostico_carpeta(DST_RES, 'RESISTENTES')
print('=' * 55)
print(f'\nTOTAL archivos útiles : {len(utiles_s) + len(utiles_r)}')
print(f'TOTAL vacíos          : {len(vacios_s) + len(vacios_r)}')
if vacios_s or vacios_r:
    print('\n⚠️  Los vacíos NO serán cargados en la siguiente celda.')



🔬 SUSCEPTIBLES
   Archivos totales     : 3228
   ✅ Con datos (útiles) : 3228
   🗑  Vacíos (≤500b)  : 0
   Genomas únicos (GCA/F): 191

🔬 RESISTENTES
   Archivos totales     : 2261
   ✅ Con datos (útiles) : 2247
   🗑  Vacíos (≤500b)  : 14
   Genomas únicos (GCA/F): 260

TOTAL archivos útiles : 5475
TOTAL vacíos          : 14

⚠️  Los vacíos NO serán cargados en la siguiente celda.


## Celda 4 · Cargar secuencias

- Fuente única: `dataset_limpio/Susceptible` y `dataset_limpio/Resistente`
- Solo `Cut_Off ∈ {Perfect, Strict}`
- Registra el `genome_id` de cada cepa para el split posterior
- Archivos ≤ `MIN_BYTES` se saltan automáticamente


In [ ]:
import pandas as pd
from tqdm import tqdm

def cargar_con_cepa(ruta, label, min_bytes=MIN_BYTES, cutoff_ok=CUTOFF_OK, min_len=MIN_SEQ_LEN):
    registros = []
    archivos  = [f for f in os.listdir(ruta) if f.endswith(('.txt', '.tsv'))]
    vacios_saltados = sin_predicted_dna = sin_hits_strict = 0
    lbl_char = 'S' if label == 0 else 'R'
    for f in tqdm(archivos, desc=lbl_char):
        ruta_f = os.path.join(ruta, f)
        # Saltar vacíos
        if os.path.getsize(ruta_f) <= min_bytes:
            vacios_saltados += 1
            continue
        # Extraer genome_id
        m = re.search(r'(GC[AF]_\d+\.\d+)', f)
        genome_id = m.group(1) if m else f.replace('_rgi.txt', '').replace('.tsv', '')
        try:
            df = pd.read_csv(ruta_f, sep='\t')
            if 'Predicted_DNA' not in df.columns:
                sin_predicted_dna += 1
                continue
            df = df[df['Cut_Off'].isin(cutoff_ok)]
            if df.empty:
                sin_hits_strict += 1
                continue
            for seq in df['Predicted_DNA'].dropna().str.strip():
                if len(seq) > min_len:
                    registros.append({
                        'sequence'  : seq.upper(),
                        'label'     : label,
                        'genome_id' : genome_id,
                        'archivo'   : f,
                    })
        except Exception as e:
            continue
    print(f'  Vacíos saltados: {vacios_saltados} | Sin Predicted_DNA: {sin_predicted_dna} | '
          f'Sin hits Strict/Perfect: {sin_hits_strict}')
    return registros

print('Cargando Susceptibles...')
d_S = cargar_con_cepa(DST_SUS, label=0)
print('Cargando Resistentes...')
d_R = cargar_con_cepa(DST_RES, label=1)

dataset = pd.DataFrame(d_S + d_R)

print(f'\n📊 Dataset crudo cargado')
print(f'   Total filas           : {len(dataset):,}')
print(f'   Secuencias únicas     : {dataset["sequence"].nunique():,}')
cepas = dataset[["genome_id","label"]].drop_duplicates()
print(f'   Cepas únicas S        : {(cepas["label"]==0).sum()}')
print(f'   Cepas únicas R        : {(cepas["label"]==1).sum()}')
print(dataset['label'].value_counts().rename({0:'Susceptible',1:'Resistente'}).to_string())


Cargando Susceptibles...


S: 100%|██████████| 3228/3228 [02:32<00:00, 21.13it/s] 


  Vacíos saltados: 0 | Sin Predicted_DNA: 0 | Sin hits Strict/Perfect: 0
Cargando Resistentes...


R: 100%|██████████| 2261/2261 [03:08<00:00, 12.00it/s]


  Vacíos saltados: 14 | Sin Predicted_DNA: 0 | Sin hits Strict/Perfect: 0

📊 Dataset crudo cargado
   Total filas           : 197,251
   Secuencias únicas     : 10,609
   Cepas únicas S        : 3055
   Cepas únicas R        : 2228
label
Susceptible    113441
Resistente      83810


## Celda 5 · Eliminar secuencias con label contradictorio

Una misma secuencia de DNA que aparece con label 0 (S) y label 1 (R) a la vez
no puede entrenarse correctamente — la descartamos.


In [ ]:
# Detectar contradictorios
conteo_labels = dataset.groupby('sequence')['label'].nunique()
seqs_ambiguas = conteo_labels[conteo_labels > 1].index

n_ambiguas     = len(seqs_ambiguas)
filas_afectadas = dataset['sequence'].isin(seqs_ambiguas).sum()
print(f'Secuencias con label contradictorio : {n_ambiguas:,}')
print(f'Filas afectadas                     : {filas_afectadas:,} ({filas_afectadas/len(dataset)*100:.1f}%)')

# Eliminar
dataset_limpio = dataset[~dataset['sequence'].isin(seqs_ambiguas)].copy().reset_index(drop=True)

print(f'\nANTES  : {len(dataset):,} filas')
print(f'DESPUÉS: {len(dataset_limpio):,} filas  (eliminadas {len(dataset)-len(dataset_limpio):,})')
print(dataset_limpio['label'].value_counts().rename({0:'Susceptible',1:'Resistente'}).to_string())


Secuencias con label contradictorio : 2,052
Filas afectadas                     : 175,635 (89.0%)

ANTES  : 197,251 filas
DESPUÉS: 21,616 filas  (eliminadas 175,635)
label
Susceptible    17548
Resistente      4068


## Celda 6 · Split Train / Validation **por cepa**

El split se hace sobre `genome_id`, no sobre secuencias.
Así ninguna cepa aparece en train y val a la vez.
El overlap de *secuencias* entre splits puede ser > 0 y es **biológicamente normal**
(el mismo gen aparece en cepas distintas).


In [ ]:
import pandas as pd
import os
from pathlib import Path

# ── CARGA DEL MANIFIESTO ST-BLOCKED ──
posibles_rutas = [
    '../data/dataset_manifest_colab.csv',
    '../data/dataset_manifest_colab.csv',
    os.path.join('..', 'data', 'dataset_manifest_colab.csv'),
    os.path.join('data', 'dataset_manifest_colab.csv'),
]

manifest_path = next((p for p in posibles_rutas if os.path.exists(p)), None)

if manifest_path is None:
    print(f'⚠️ ERROR: dataset_manifest_colab.csv no encontrado en ninguna ruta: {posibles_rutas}')
else:
    manifest = pd.read_csv(manifest_path)
    print(f'✅ Manifiesto cargado exitosamente desde: {manifest_path}')
    
    # Unir dataset_limpio con el manifiesto para traer el ST y la particion
    dataset_limpio = dataset_limpio.merge(
        manifest[['original_filename', 'sequence_type', 'dataset_partition']], 
        left_on='archivo', 
        right_on='original_filename', 
        how='inner'
    )

    train_df = dataset_limpio[dataset_limpio['dataset_partition'] == 'train'].reset_index(drop=True)
    val_df   = dataset_limpio[dataset_limpio['dataset_partition'] == 'validation'].reset_index(drop=True)

    train_ids = set(train_df['genome_id'])
    val_ids   = set(val_df['genome_id'])

    print('SPLIT ST-BLOCKED APLICADO DESDE MANIFIESTO')
    print(f'  Train : {len(train_df):,} secuencias de {len(train_ids):,} cepas')
    print(f'  Val   : {len(val_df):,} secuencias de {len(val_ids):,} cepas')
    print(f'  Distribución train : {train_df["label"].value_counts().to_dict()}')
    print(f'  Distribución val   : {val_df["label"].value_counts().to_dict()}')


SPLIT POR CEPA
  Train : 17,112 secuencias de 3,078 cepas
  Val   : 4,504 secuencias de 770 cepas
  Distribución train : {0: 13807, 1: 3305}
  Distribución val   : {0: 3741, 1: 763}


## Celda 7 · Auditoría completa

Verifica leakage, distribuciones, proxys y cruces entre fuentes.


In [ ]:
import numpy as np

RED = '\033[91m'; GRN = '\033[92m'; YEL = '\033[93m'; RST = '\033[0m'
alertas = []
oks     = []

if 'sequence_type' not in train_df.columns:
    print(f"{RED}⚠️ ERROR: La columna 'sequence_type' no existe. Asegúrate de haber subido el manifiesto.{RST}")
else:
    # CHECK 1 — ST Overlap (DEBE SER 0 para evitar data leakage clonal)
    st_train = set(train_df['sequence_type'])
    st_val = set(val_df['sequence_type'])
    overlap_st = st_train.intersection(st_val)

    if overlap_st:
        alertas.append(f'LEAKAGE ST DETECTADO: {len(overlap_st)} STs solapados entre train y val: {overlap_st}')
    else:
        oks.append(f'ST Overlap = 0. Sin leakage clonal confirmado (Train STs: {len(st_train)}, Val STs: {len(st_val)})')

# CHECK 2 — Overlap de cepas (DEBE SER 0)
overlap_cepas = train_ids & val_ids
if overlap_cepas:
    alertas.append(f'OVERLAP DE CEPAS: {len(overlap_cepas)} cepas en train Y val')
else:
    oks.append('Sin overlap de cepas entre train y val')

# CHECK 3 — Balance de clases train vs val
tr_dist  = train_df['label'].value_counts(normalize=True).sort_index()
val_dist = val_df['label'].value_counts(normalize=True).sort_index()
diff = (tr_dist - val_dist).abs().max()
if diff > 0.05:
    alertas.append(f'DESBALANCE ENTRE SPLITS: diferencia {diff:.3f} en distribución de clases')
else:
    oks.append(f'Distribución de clases consistente (diff={diff:.4f})')

# CHECK 4 — Proxy leakage: longitud de secuencia
dataset_limpio['_seq_len'] = dataset_limpio['sequence'].str.len()
corr = dataset_limpio['_seq_len'].corr(dataset_limpio['label'])
dataset_limpio.drop(columns=['_seq_len'], inplace=True)
if abs(corr) > 0.3:
    alertas.append(f'PROXY LEAKAGE: longitud de secuencia correlaciona {corr:.3f} con label')
else:
    oks.append(f'Longitud de secuencia no es proxy del label (r={corr:.3f})')

# CHECK 5 — Duplicados en dataset_limpio
dupes = dataset_limpio.duplicated(subset='sequence').sum()
if dupes > 0:
    alertas.append(f'DUPLICADOS: {dupes:,} secuencias repetidas en dataset_limpio (normal si el mismo gen aparece en varias cepas)')
else:
    oks.append('Sin secuencias duplicadas en dataset_limpio')

# ── RESUMEN ──────────────────────────────────────────────────────────────
print('\n' + '='*60)
for msg in oks:
    print(f'{GRN}✓ {msg}{RST}')
if alertas:
    print(f'\n{RED}🚨 {len(alertas)} ALERTAS:{RST}')
    for i, a in enumerate(alertas, 1):
        print(f'{RED}  [{i}] {a}{RST}')
else:
    print(f'\n{GRN}✅ Auditoría superada sin alertas críticas.{RST}')
print('='*60)


ℹ️  Overlap secuencias train∩val: 1,407 (52.0% del val) — biológicamente normal

✓ Sin overlap de cepas entre train y val
✓ Split cubre el 100% del dataset limpio
✓ Distribución de clases consistente (diff=0.0237)
✓ Longitud de secuencia no es proxy del label (r=-0.020)

🚨 1 ALERTAS:
  [1] DUPLICADOS: 13,059 secuencias repetidas en dataset_limpio (normal si el mismo gen aparece en varias cepas)


## Celda 8 · Guardar CSV finales

Guarda `train_por_cepa.csv` y `val_por_cepa.csv` en `dataset_limpio/`.
Si ya existen, los sobreescribe con la versión actual (limpia y auditada).


In [ ]:
ruta_train = os.path.join(RUTA_SALIDA, 'train_por_cepa.csv')
ruta_val   = os.path.join(RUTA_SALIDA, 'val_por_cepa.csv')

train_df.to_csv(ruta_train, index=False)
val_df.to_csv(ruta_val,   index=False)

print('✅ Archivos guardados:')
print(f'   {ruta_train}')
print(f'   {ruta_val}')
print()
print('RESUMEN FINAL')
print('=' * 55)
print(f'  Dataset total (limpio)  : {len(dataset_limpio):,} secuencias')
print(f'  Cepas únicas S          : {(cepas["label"]==0).sum()}')
print(f'  Cepas únicas R          : {(cepas["label"]==1).sum()}')
print(f'  Train  ({int((1-VAL_FRACTION)*100)}%)          : {len(train_df):,} seqs | {len(train_ids)} cepas')
print(f'  Val    ({int(VAL_FRACTION*100)}%)           : {len(val_df):,} seqs | {len(val_ids)} cepas')
print('=' * 55)


✅ Archivos guardados:
   ../data/train_por_cepa.csv
   ../data/val_por_cepa.csv

RESUMEN FINAL
  Dataset total (limpio)  : 21,616 secuencias
  Cepas únicas S          : 2399
  Cepas únicas R          : 1449
  Train  (80%)          : 17,112 seqs | 3078 cepas
  Val    (20%)           : 4,504 seqs | 770 cepas
